# Classificatore basato su SVM lineari e word embedding

In questo notebook viene svolto il terzo punto del progetto: classificatore con rappresentazione del testo vettoriale costruita a partire da word embeddings. Gli embeddings utilizzati sono i seguenti: itWaC: billion word corpus constructed from the Web limiting the crawl to the .it domain and using medium-frequency words from the Repubblica corpus and basic Italian vocabulary lists as seeds

In [1]:
import pandas as pd
import numpy as np
import stanza
import importlib
import Classes
importlib.reload(Classes)
from Classes import *
from sklearn.feature_extraction import DictVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
from sklearn.preprocessing import MaxAbsScaler
from sklearn.metrics import accuracy_score, f1_score

In [2]:
import re

In [3]:
import sqlite3

In [4]:
train_documents = pd.read_pickle("train_documents.pkl")
val_documents = pd.read_pickle("val_documents.pkl")
test_documents = pd.read_pickle("test_documents.pkl")

### Carico gli embeddings

In [5]:
embeddings = load_embeddings_from_sqlite("embeddits.sqlite")

In [6]:
print(f"Numero parole nel vocabolario: {len(embeddings):,}")
print(f"Dimensione embedding: {len(embeddings['casa'])}")
print(f"Embedding 'casa': {embeddings['casa']}")

Numero parole nel vocabolario: 1,247,492
Dimensione embedding: 128
Embedding 'casa': [ 1.98289305e-02 -2.99291890e-02  1.07659526e-01 -1.23271987e-01
  1.24683408e-02 -5.89558221e-02  9.13426504e-02 -2.13298231e-01
 -5.62658943e-02  6.76765805e-03  1.23197539e-02 -2.48114560e-02
 -4.40169908e-02 -2.04461403e-02  6.20883666e-02 -2.14107428e-03
 -8.04886445e-02 -2.37802416e-02  9.46704485e-03  1.01404823e-01
 -1.96930587e-01  4.75535281e-02  2.47208364e-02 -8.94748047e-02
 -1.69130370e-01  4.99142222e-02 -8.32871795e-02 -1.15884067e-02
  5.67554235e-02  4.03301530e-02 -5.73197287e-03 -3.97101976e-02
  3.35753895e-02  1.41663924e-01  8.48880261e-02  4.97753732e-02
 -1.78466756e-02  6.92231953e-02  1.17348907e-04 -1.31110325e-02
 -8.82359594e-02 -1.03949765e-02  2.06487812e-02 -1.10362872e-01
 -3.29698145e-01  1.78224489e-01  3.36985439e-02  8.64214078e-02
 -1.55117884e-01 -1.65368870e-01  2.10813954e-01 -2.54938025e-02
 -5.51235378e-02  6.61397800e-02 -1.03666030e-01 -3.94498482e-02
  2.1

### Normalizzo il testo

In [7]:
for doc in train_documents:
    for token in doc.tokens:
        token.word = normalize_text(token.word)

for doc in val_documents:
    for token in doc.tokens:
        token.word = normalize_text(token.word)

for doc in test_documents:
    for token in doc.tokens:
        token.word = normalize_text(token.word)

### Metodi di aggregazione

In [8]:
embeddings_dim = 128

#  Funzione che calcola la media per colonna degli embedding. 

def compute_embeddings_mean(document_embeddings):
    if len(document_embeddings) > 0:
        return np.mean(document_embeddings, axis=0)
    else:
        return np.zeros(embeddings_dim)

In [9]:
# media di tutti gli embedding del documento

def compute_all_embeddings_mean(document):
    document_embeddings = []
    
    for word in document.get_words():
        if word in embeddings:
            document_embeddings.append(embeddings[word])
    
    return compute_embeddings_mean(document_embeddings)


# media dei word embedding di parole piene

def compute_filtered_embeddings_mean(document):
    document_embeddings = []
    
    for token in document.get_tokens():
        word = token.word
        pos = token.pos
        if word in embeddings and pos in ['ADJ', 'NOUN', 'VERB']:
            document_embeddings.append(embeddings[word])
    
    return compute_embeddings_mean(document_embeddings)


# media separata dei word embedding di parole piene

def compute_filtered_embeddings_sep_means(document):
    target_pos = ['ADJ', 'NOUN', 'VERB']
    document_embeddings = {pos: [] for pos in target_pos}

    for token in document.get_tokens():
        word = token.word
        pos = token.pos
        if word in embeddings and pos in target_pos:
            document_embeddings[pos].append(embeddings[word])

    mean_document_embeddings = []
    for pos in target_pos:
        mean_document_embeddings.append(compute_embeddings_mean(document_embeddings[pos]))

    mean_document_embeddings = np.concatenate(mean_document_embeddings, axis=0)
    return mean_document_embeddings

In [10]:
# strategia 1: media di tutti i token
X_train_all = np.array([compute_all_embeddings_mean(doc) for doc in train_documents])
X_val_all = np.array([compute_all_embeddings_mean(doc) for doc in val_documents])
X_test_all = np.array([compute_all_embeddings_mean(doc) for doc in test_documents])

In [11]:
# strategia 2: media solo di ADJ/NOUN/VERB
X_train_filt = np.array([compute_filtered_embeddings_mean(doc) for doc in train_documents])
X_val_filt = np.array([compute_filtered_embeddings_mean(doc) for doc in val_documents])
X_test_filt = np.array([compute_filtered_embeddings_mean(doc) for doc in test_documents])

In [12]:
# strategia 3: media separata per POS + concatenazione
X_train_sep = np.array([compute_filtered_embeddings_sep_means(doc) for doc in train_documents])
X_val_sep = np.array([compute_filtered_embeddings_sep_means(doc) for doc in val_documents])
X_test_sep = np.array([compute_filtered_embeddings_sep_means(doc) for doc in test_documents])

### Training modello su metodo 1

In [13]:
labels = [doc.label for doc in train_documents]
labels_val = [doc.label for doc in val_documents]
labels_t = [doc.label for doc in test_documents]

In [14]:
model_all = LinearSVC(random_state=42, max_iter=5000)
model_all.fit(X_train_all, labels)

LinearSVC(max_iter=5000, random_state=42)

In [15]:
print(classification_report(labels_val, model_all.predict(X_val_all)))

              precision    recall  f1-score   support

           0       0.92      0.93      0.93       500
           1       0.93      0.92      0.93       500

    accuracy                           0.93      1000
   macro avg       0.93      0.93      0.93      1000
weighted avg       0.93      0.93      0.93      1000



### Training modello su metodo 2

In [16]:
model_filt = LinearSVC(random_state=42, max_iter=5000)
model_filt.fit(X_train_filt, labels)

LinearSVC(max_iter=5000, random_state=42)

In [17]:
print(classification_report(labels_val, model_filt.predict(X_val_filt)))

              precision    recall  f1-score   support

           0       0.90      0.90      0.90       500
           1       0.90      0.90      0.90       500

    accuracy                           0.90      1000
   macro avg       0.90      0.90      0.90      1000
weighted avg       0.90      0.90      0.90      1000



### Training modello su metodo 3

In [18]:
model_sep = LinearSVC(random_state=42, max_iter=5000)
model_sep.fit(X_train_sep, labels)

LinearSVC(max_iter=5000, random_state=42)

In [19]:
print(classification_report(labels_val, model_sep.predict(X_val_sep)))

              precision    recall  f1-score   support

           0       0.92      0.94      0.93       500
           1       0.94      0.92      0.93       500

    accuracy                           0.93      1000
   macro avg       0.93      0.93      0.93      1000
weighted avg       0.93      0.93      0.93      1000



In [21]:
print(f"{'Strategia':<30} {'F1 macro (val)':>15}")
print("-" * 45)
print(f"{'1. Media tutti gli embedding':<30} {f1_score(labels_val, model_all.predict(X_val_all), average='macro'):>15.3f}")
print(f"{'2. Media solo ADJ/NOUN/VERB':<30} {f1_score(labels_val, model_filt.predict(X_val_filt), average='macro'):>15.3f}")
print(f"{'3. Media separata + concat':<30} {f1_score(labels_val, model_sep.predict(X_val_sep), average='macro'):>15.3f}")

Strategia                       F1 macro (val)
---------------------------------------------
1. Media tutti gli embedding             0.926
2. Media solo ADJ/NOUN/VERB              0.903
3. Media separata + concat               0.929


### Test 

In [20]:
print(classification_report(labels_t, model_sep.predict(X_test_sep)))

              precision    recall  f1-score   support

           0       0.73      0.94      0.82       500
           1       0.92      0.65      0.76       500

    accuracy                           0.80      1000
   macro avg       0.83      0.80      0.79      1000
weighted avg       0.83      0.80      0.79      1000

